# **Bronze Layer - Performance Data Ingestion**

- **Project:** Mortgage Portfolio Product Governance & Performance Review
- **Layer:** Bronze (raw landing)
- **Source:** Freddie Mac Single-Family Loan-Level Dataset - 2018 performance sample (Pre-July 2026 layout)

This notebook lands the raw Freddie Mac performance file into the Lakehouse as a
Delta table ("bronze_performance"). It applies column names from the confirmed
file layout, adds lineage columns for governance, and reconciles the load before
recording the result in an audit table.

**What it does, step by step:**
1. Reads the pipe-delimited raw file, applying 32 column names in file order (all as text, to preserve source values faithfully)
2. Adds lineage columns - load batch, ingestion timestamp, source file
3. Writes the result to the "bronze_performance" Delta table
4. Reconciles raw line count against loaded rows to prove no data was lost during ingestion
5. Records the reconciliation outcome in a "load_audit" table

**Design principles:** Faithful bronze landing (type later in dbt) and auditable
loads (every load is lineage-tagged and reconciled).


### 1. Read and name columns

In [1]:
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql import Row

# applied column names to the raw file
# From schema/performance.py - 32 columns, exact file order (Pre-July 2026 layout)
performance_columns = [
    "loan_sequence_number", "monthly_reporting_period", "current_actual_upb", 
    "current_loan_delinquency_status", "loan_age", "remaining_months_to_legal_maturity", 
    "defect_settlement_date", "modification_flag", "zero_balance_code", "zero_balance_effective_date", 
    "current_interest_rate", "current_deferred_upb", "ddlpi", "mi_recoveries", "net_sales_proceeds", 
    "non_mi_recoveries", "expenses", "legal_costs", "maintenance_and_preservation_costs", 
    "taxes_and_insurance", "miscellaneous_expenses", "actual_loss_calculation", "modification_cost", 
    "step_modification_flag", "deferred_payment_plan", "eltv", "zero_balance_removal_upb", 
    "delinquent_accrued_interest", "delinquency_due_to_disaster", "borrower_assistance_status_code", 
    "current_month_modification_cost", "interest_bearing_upb",
]

raw_path = "Files/raw/performance/sample_svcg_2018.txt"

df = (spark.read
    .option("sep","|")
    .option("header","false")
    .option("inferSchema","false") # keep everything as strings - faithful bronze layer
    .csv(raw_path)
    .toDF(*performance_columns)) # apply names positionally

df.printSchema()

StatementMeta(, bde685e1-ab9d-4ad6-ae27-5fe5f6005a70, 3, Finished, Available, Finished, False)

root
 |-- loan_sequence_number: string (nullable = true)
 |-- monthly_reporting_period: string (nullable = true)
 |-- current_actual_upb: string (nullable = true)
 |-- current_loan_delinquency_status: string (nullable = true)
 |-- loan_age: string (nullable = true)
 |-- remaining_months_to_legal_maturity: string (nullable = true)
 |-- defect_settlement_date: string (nullable = true)
 |-- modification_flag: string (nullable = true)
 |-- zero_balance_code: string (nullable = true)
 |-- zero_balance_effective_date: string (nullable = true)
 |-- current_interest_rate: string (nullable = true)
 |-- current_deferred_upb: string (nullable = true)
 |-- ddlpi: string (nullable = true)
 |-- mi_recoveries: string (nullable = true)
 |-- net_sales_proceeds: string (nullable = true)
 |-- non_mi_recoveries: string (nullable = true)
 |-- expenses: string (nullable = true)
 |-- legal_costs: string (nullable = true)
 |-- maintenance_and_preservation_costs: string (nullable = true)
 |-- taxes_and_insuran

### 2. Inspect and sanity-check

In [2]:
display(df.limit(10)) # first 10 rows
print("Row count:", df.count())

StatementMeta(, bde685e1-ab9d-4ad6-ae27-5fe5f6005a70, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 629ca71f-aac5-40b5-9971-9153a04f388a)

Row count: 1998728


## **Data Governance**

### 3. Add lineage columns

In [3]:
# added lineage columns for governance
df_bronze = (df
    .withColumn("load_batch_id", lit("2018_svcg_001"))
    .withColumn("ingested_at_utc", current_timestamp())
    .withColumn("source_file", lit(raw_path)))

display(df_bronze.limit(3))

StatementMeta(, bde685e1-ab9d-4ad6-ae27-5fe5f6005a70, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fec0b47d-cd23-453e-a734-d77108addfc5)

### 4. Write to the bronze Delta table

In [4]:
# wrote df_broze as a delta table in the lakehouse
(df_bronze.write
.mode("overwrite")
.format("delta")
.saveAsTable("bronze_performance"))

StatementMeta(, bde685e1-ab9d-4ad6-ae27-5fe5f6005a70, 6, Finished, Available, Finished, False)

### 5. Reconcile the load

In [6]:
# reconcilation checks - to confirm raw data and loaded dataframe row count matches
raw_line_count = spark.read.text(raw_path).count()
loaded_count = spark.table("bronze_performance").count()

print("Raw count:", raw_line_count)
print("Loaded count:", loaded_count)
print(raw_line_count == loaded_count)

StatementMeta(, bde685e1-ab9d-4ad6-ae27-5fe5f6005a70, 8, Finished, Available, Finished, False)

Raw count: 1998728
Loaded count: 1998728
True


### 6. Record the result in the audit table

In [10]:
# wrote the reconciliation result to a load-audit table
audit = spark.createDataFrame([Row(
    load_batch_id = "2018_svcg_001",
    table_name = "bronze_performance",
    source_file = raw_path,
    raw_line_count = raw_line_count,
    loaded_count = loaded_count,
    reconciled = bool(raw_line_count == loaded_count)
)])

# display(audit)

# wrote the reconciliation result to a load-audit table
(audit.write
.mode("append")
.format("delta")
.saveAsTable("load_audit"))

display(spark.table("load_audit"))

StatementMeta(, bde685e1-ab9d-4ad6-ae27-5fe5f6005a70, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9f2138b8-8eaf-475f-9eb5-9ada477ef2c3)